# Beam Expander Design

In [ ]:
import numpy as np

from optiland import optic, optimization
from optiland.optimization import minimize

Define a starting lens:

In [ ]:
lens = optic.Optic()

# define beam expander properties
radius_start = 5
radius_end = 15

# add surfaces
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(index=1, thickness=5, radius=-50, material="N-BK7", is_stop=True)
lens.surfaces.add(index=2, thickness=75, radius=np.inf)
lens.surfaces.add(index=3, thickness=5, radius=np.inf, material="N-BK7")
lens.surfaces.add(index=4, thickness=50, radius=-300)
lens.surfaces.add(index=5)

# set aperture
lens.set_aperture(aperture_type="EPD", value=2 * radius_start)

# add field
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)

# add wavelength
lens.wavelengths.add(value=0.633, is_primary=True)

# draw lens
lens.draw(num_rays=5)

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization):

In [ ]:
# Ray y-intersect at last lens surface
input_data = {
    "optic": lens,
    "surface_number": 4,
    "Hx": 0,
    "Hy": 0,
    "Px": 0,
    "Py": 1,
    "wavelength": 0.633,
}
problem.add_operand(
    operand_type="real_y_intercept",
    target=radius_end,
    weight=1,
    input_data=input_data,
)

# Ray y-intersect at image surface
input_data = {
    "optic": lens,
    "surface_number": 5,
    "Hx": 0,
    "Hy": 0,
    "Px": 0,
    "Py": 1,
    "wavelength": 0.633,
}
problem.add_operand(
    operand_type="real_y_intercept",
    target=radius_end,
    weight=1,
    input_data=input_data,
)

Define variables - let two radii of curvature vary:

In [ ]:
problem.add_variable(lens, "radius", surface_number=1)  # first surface of first lens
problem.add_variable(lens, "radius", surface_number=4)  # second surface of second lens

Check initial merit function value and system properties:

In [ ]:
problem.info()

Run optimization:

In [ ]:
result = minimize(problem, "dls", tol=1e-6)

Run optimization:

In [ ]:
print(result)

Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
lens.draw(num_rays=5)